In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path("data/processed/ChargingRecords_cleaned.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("ChargingRecords_features.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/processed/ChargingRecords_features.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Cleaned or feature-engineered dataset not found.")

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Loaded: data\processed\ChargingRecords_features.csv
Shape: (72799, 32)


,UserID,ChargerID,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,...,Location_golf,Location_hotel,Location_market,Location_public area,Location_public institution,Location_public parking lot,Location_resort,Location_restaurant,Location_sightseeing,ChargerType_1
0,0,1,15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36,...,0,1,0,0,0,0,0,0,0,0
1,0,1,14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,2022-09-14 21:31:04,90,10.19,...,0,1,0,0,0,0,0,0,0,0
2,0,1,14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,2022-09-14 19:54:29,60,6.78,...,0,1,0,0,0,0,0,0,0,0
3,0,1,29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,2022-09-30 00:16:42,344,37.65,...,0,1,0,0,0,0,0,0,0,0
4,0,1,25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:15,2022-09-26 00:30:14,300,33.81,...,0,1,0,0,0,0,0,0,0,0


In [9]:
TARGET = "Demand"

if TARGET not in df.columns:
    raise ValueError("Demand column not found.")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()

exclude_columns = [
    "Demand",
    "DemandCategory",
    "UserID",
    "ChargerID",
    "StartDatetime",
    "EndDatetime",
    "EndDay",
    "EndTime",
    "Duration",
    "DurationHours"
]

X = df.drop(
    columns=[c for c in exclude_columns if c in df.columns]
)
y = df[TARGET]

# Remove columns with no usable values
X = X.dropna(axis=1, how="all")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", X.columns.tolist())

Feature shape: (72799, 22)
Target shape: (72799,)
Features: ['StartDay', 'StartTime', 'StartHour', 'StartDayOfWeek', 'StartMonth', 'IsWeekend', 'StartDayName', 'ChargerCompany_1', 'Location_apartment', 'Location_bus garage', 'Location_camping', 'Location_company', 'Location_golf', 'Location_hotel', 'Location_market', 'Location_public area', 'Location_public institution', 'Location_public parking lot', 'Location_resort', 'Location_restaurant', 'Location_sightseeing', 'ChargerType_1']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 58239
Testing rows: 14560


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number", "bool"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Memory-efficient preprocessing ready!")

Memory-efficient preprocessing ready!


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=30,
        max_depth=10,
        max_features=0.7,
        random_state=42,
        n_jobs=2
    ))
])

print("Training model...")

model.fit(X_train, y_train)

predictions = model.predict(X_test)

results_df = pd.DataFrame([{
    "Model": "Random Forest",
    "MAE": mean_absolute_error(y_test, predictions),
    "MSE": mean_squared_error(y_test, predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
    "R2": r2_score(y_test, predictions)
}])

trained_models = {"Random Forest": model}

print("Training completed successfully!")
display(results_df)

Training model...
Training completed successfully!


,Model,MAE,MSE,RMSE,R2
0,Random Forest,9.69318,161.971093,12.726786,0.111391


In [13]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

predictions = best_model.predict(X_test)

comparison_df = pd.DataFrame({
    "Actual_Demand": y_test.to_numpy(),
    "Predicted_Demand": predictions
})

print("Best model:", best_model_name)
display(comparison_df.head(10))

Best model: Random Forest


,Actual_Demand,Predicted_Demand
0,14.60,18.814025
1,31.09,18.851413
2,18.23,18.278809
3,5.30,17.244606
4,0.82,16.675671
5,21.20,16.503464
6,0.03,18.563975
7,25.07,23.098235
8,44.10,14.506884
9,20.50,14.893810


In [14]:
import joblib

OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_model,
    OUTPUT_DIR / "best_demand_regression_model.joblib"
)

results_df.to_csv(
    REPORT_DIR / "regression_model_results.csv",
    index=False
)

comparison_df.to_csv(
    REPORT_DIR / "regression_predictions.csv",
    index=False
)

print("Regression training completed!")
print("Best model:", best_model_name)
print("Model saved in:", OUTPUT_DIR.resolve())
print("Reports saved in:", REPORT_DIR.resolve())

Regression training completed!
Best model: Random Forest
Model saved in: C:\Users\kavee\OneDrive\Desktop\EV Charging Demand Prediction and Usage Pattern Analysis\notebook\models
Reports saved in: C:\Users\kavee\OneDrive\Desktop\EV Charging Demand Prediction and Usage Pattern Analysis\notebook\reports
